# Homework 4
# Perceptron, SVM, and PCA

# <p style="text-align: right;"> &#9989; Maanya Sharma</p>
# <p style="text-align: right;"> &#9989; maanyasharma</p>

# Goal for this homework assignment
We have worked some basics on perceptron, SVM, and PCA in the pre-class and in-class assignments. In this homework assignment, we will:

* Continue to use git as the version control tool
* Work on unfamiliar data
* Use perceptron to classify data 
* Use SVM to classify data
* Use principal component analysis to facilitate classification


**This assignment is due by 11:59 pm on Friday, April 25th. Note that ONLY the copy on GITHUB will be graded.**  **There are 60 standard points possible in this assignment including points for Git commits/pushes. The distribution of points can be found in the section headers**.

---
# Part 1: Git repository (6 points)

You're going to add this assignment to the `cmse202-s25-turnin` repository you previously created. The history of progress on the assignment will be tracked via git commitments. 

**&#9989; Do the following**:

1. Navigate to your `cmse202-s25-turnin` **local** repository and create a new directory called `hw-04`

2. Move this notebook into that **new directory** in your repository. 

5. Double check to make sure your file is at the correct directory.

6. Once you're certain that file and directory are correct, add this notebook to your repository, then make a commit and push it to GitHub. You may need to use `git push origin hw04` to push your file to GitHub.

Finally, &#9989; **Do this**: Before you move on, put the command that your instructor should run to clone your repository in the markdown cell below. **Points for this part will be given for correctly setting up branch, etc., above, and for doing git commits/pushes mentioned throughout the assignment.**

<font size=6 color="#009600">&#9998;</font>git clone -b hw-04 https://github.com/maanyasharma/cmse202-s25-turnin.git

**Important**: Double check you've added your Professor and your TA as collaborators to your "turnin" repository (you should have done this in the previous homework assignment).

**Also important**: Make sure that the version of this notebook that you are working on is the same one that you just added to your repository! If you are working on a different copy of the notebook, **none of your changes will be tracked**!

If everything went as intended, the file should now show up on your GitHub account in the "`cmse202-s25-turnin`" repository inside the `hw-04` directory that you just created.

Periodically, **you'll be asked to commit your changes to the repository and push them to the remote GitHub location**. Of course, you can always commit your changes more often than that, if you wish.  It can be good to get into a habit of committing your changes any time you make a significant modification, or when you stop working on the problems for a bit.

---
# Part 2: Deal with unfamiliar data (35 points)

## Warm up with perceptron for binary classification
## 2.1 Load up the dataset

This data is obtained from Kaggle/diabetes. It contains multiple measured values and a label for whether the patient is diagnosed as diabetic. 

* Use commands to dowdload the dataset from `https://raw.githubusercontent.com/huichiayu/cmse202-s25-supllemental_data/refs/heads/main/HW04/diabetes_prediction_dataset.csv`
* Use Pandas to load in the data and briefly examine it.
* Succeed data load-up gets **2 pt**. 

In [7]:
# put your code here
import pandas as pd
url = "https://raw.githubusercontent.com/huichiayu/cmse202-s25-supplemental_data/refs/heads/main/HW04/diabetes_prediction_dataset.csv"
df = pd.read_csv(url)
df.head()



,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


How many patients are in this dataset? What are features of the patients?

<font size=6 color="#009600">&#9998;</font> There are 100,000 patients in this dataset. The features include gender, age, whether the patient has hypertension or heart disease, their smoking history, BMI, HbA1c level, blood glucose level, and whether they are diagnosed with diabetes.

### Use your perceptron class built in Day18 and Day19 assignments to classify whether patients are diabetic.

* You should see that there are some features that are non-numerics.
* The first one is `gender`. Find the types of classes and convert them to numerics in your dataframe.
* The second one is `smoking_history`, convert those string labels to numerics.
* Note that since perceptron is a binary classifier, which only determines which side of the dividing line the data points reside, we should also convert the labels to `+1` and `-1`.
* Completing data conversion gets **5 pt**.

In [ ]:
# put your code here

df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
df['smoking_history'] = df['smoking_history'].astype('category').cat.codes
df['diabetes'] = df['diabetes'].map({0: -1, 1: 1})
df.head()


### Now all feature varilables are numerics.

### &#128721; STOP (1 Point)
**Pause, save and commit your changes to your Git repository!**

Take a moment to save your notebook, commit the changes to your Git repository with a meaningful commit message.



---

## 2.2 Binary perceptron classifier

Copy your perceptron class to the cell below. 

* DO NOT use the one from statsmodel. We want to test the perceptron you built.
* Note that your predict method should output `+1` or `-1` for positive or negative values, respectively.
* A functional perceptron classifier gets **4 pt**.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class Perceptron:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
    def _unit(self, x):
        return np.where(x >= 0, 1, -1)
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_pred = self._unit(linear_output)
                update = self.lr * (y[idx] - y_pred)
                self.weights += update * x_i
                self.bias += update
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self._unit(linear_output)

if 'HbA1c level' in df.columns:
    df.rename(columns={'HbA1c level': 'HbA1c_level'}, inplace=True)

df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
df['smoking_history'] = df['smoking_history'].astype('category').cat.codes
df['diabetes'] = df['diabetes'].map({0: -1, 1: 1})

features = ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']
X = df[features].values
y = df['diabetes'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
p = Perceptron(lr=0.01, n_iters=1000)
p.fit(X_train, y_train)
y_pred = p.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)



* Split data to 70-30 train-test sets **1 pt**.
* Train your perceptron.
* Show the accuracy of your pereptron **2 pt**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
p = Perceptron(lr=0.01, n_iters=1000)
p.fit(X_train, y_train)
y_pred = p.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)



* Use test set to evaulate the accuracy of your perceptron. What is your accuracy? (**2 pt**)

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
p = Perceptron(lr=0.01, n_iters=1000)
p.fit(X_train, y_train)

train_pred = p.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)
print("Training accuracy:", train_acc)

test_pred = p.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print("Test accuracy:", test_acc)



Training accuracy: 0.9149571428571428
Test accuracy: 0.9151


* There may be some ways to increase the accruacy, such as increasing the number of train iterations or adjust learning rate. Give a try to train a perceptron you can best get. Record the values of parameters and the optimal accuracy. (**3 pt**)


In [ ]:
# put your code here
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]
n_iters_list    = [100, 500, 1000, 2000]

best_acc   = 0.0
best_lr    = None
best_iters = None

for lr in learning_rates:
    for n_iters in n_iters_list:
        model = Perceptron(lr=lr, n_iters=n_iters)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        if acc > best_acc:
            best_acc   = acc
            best_lr    = lr
            best_iters = n_iters

print(f"Best learning rate: {best_lr}")
print(f"Best iterations:    {best_iters}")
print(f"Best test accuracy: {best_acc:.4f}")


### &#128721; STOP (1 Point)
**Pause, save and commit your changes to your Git repository!**

Take a moment to save your notebook, commit the changes to your Git repository with a meaningful commit message.

---

### 2.3 Next we shall test perceptron's capability of multiple-label classification.

* Dowdload the dataset from `https://raw.githubusercontent.com/huichiayu/cmse202-s25-supllemental_data/refs/heads/main/HW04/Telecust1.csv`.
* This is a customer category dataset (Kraggle/Customer Classification). Each cusmtoer has several feature variables.
* There are five categories of customers, which are non-numerics. Thus, let's convert those string labels to numerics.
* Successful data load-up gets **2 pt**.

In [ ]:
# Download and load the dataset. Convert non-numerical labels to numerics.
# put your code here

import pandas as pd

url = "https://raw.githubusercontent.com/huichiaiyu/cmse202-s25-supplemental_data/refs/heads/main/HW04/Telecust1.csv"
df2 = pd.read_csv(url)

# convert all non-numeric (object) columns to integer codes
for col in df2.select_dtypes(include='object').columns:
    df2[col] = df2[col].astype('category').cat.codes

df2.head()


---
### 2.4 Multi-label perceptron classification

* As we know, perceptron is a binary classifier. For multiple-label classification, we can use One-vs-Rest (OvR) Strategy.
* In this case, let's train five individual perceptrons. 
* For each classifier, it treats the current class as "positive" and all others as "negative."
* When classifying a new sample, each classifier gives a "score," and the class with the highest score is chosen.

Copy your perceptron to the code cell below. We need to add a score method, which outputs dot of weights and features, as opposed to the previous binary predict method. The score method should output a signed floating score value, not `+1` or `-1`. This can be done by removing the binary segmenting, i.e., directly outputing the dot value.

* Functioning score() method gets **2 pt**.

In [ ]:

import numpy as np

class Perceptron:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
    def _unit(self, x):
        return np.where(x >= 0, 1, -1)
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_pred = self._unit(linear_output)
                update = self.lr * (y[idx] - y_pred)
                self.weights += update * x_i
                self.bias += update
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self._unit(linear_output)
    def score(self, X):
        return np.dot(X, self.weights) + self.bias


* Now let's do a train-test split of the data with a test_size = 0.3.
* Since we are training 5 perceptrons, we should have have 5 class label sets. For instance, in the label set for category A, the label value will be `+1` if it's type A and otherwise `-1`.
* Setting label sets gets **4 pt**.

In [ ]:
# put your code here
import numpy as np
from sklearn.model_selection import train_test_split

X = df2.iloc[:, :-1].values
y = df2.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

classes = np.unique(y_train)

y_train_sets = {c: np.where(y_train == c, 1, -1) for c in classes}
y_test_sets  = {c: np.where(y_test  == c, 1, -1) for c in classes}


* Use training set and the 5 training label sets to train your 5 perceptrons. Report the accuracy of those five training.
* Efficiently train the five perceptrons using nest loop gets **5 pt**.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

classes = np.unique(y_train)

y_train_sets = {c: np.where(y_train == c, 1, -1) for c in classes}

perceptrons     = {}
train_accuracies = {}

for c in classes:
    p = Perceptron(lr=0.01, n_iters=1000)
    p.fit(X_train, y_train_sets[c])
    preds = p.predict(X_train)
    acc   = accuracy_score(y_train_sets[c], preds)
    perceptrons[c]     = p
    train_accuracies[c] = acc

for c, acc in train_accuracies.items():
    print(f"Class {c}: Training accuracy = {acc:.4f}")


* Use the test vector to examine the accuracy.
* For each feature set, there should be 5 output scores, each from a perceptron. The predicted label should be the label that corresponds to the highest score.
* Report your accuracy. (**3 pt**)

In [ ]:
# put your code here
import numpy as np
from sklearn.metrics import accuracy_score

# assemble the test‐set scores: one column per class
score_matrix = np.column_stack([
    perceptrons[c].score(X_test)
    for c in classes
])

# pick the class with the highest score
y_pred = classes[np.argmax(score_matrix, axis=1)]

# compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Multi-label test accuracy:", accuracy)


How good is your multiple-label perceptron classification?

<font size=6 color="#009600">&#9998;</font> I got about a 68 % hit rate—much better than the 20 % you'd see by chance, but still missing a third of the customers whose profiles overlap. Linear perceptrons do a solid job on the big-picture segments, but you’d need richer features or a non-linear model to tease apart the finer distinctions.


### &#128721; STOP (1 Point)
**Pause, save and commit your changes to your Git repository!**

Take a moment to save your notebook, commit the changes to your Git repository with a meaningful commit message.

---
## Part 3 SVM classifiers (19 points)

### 3.1 SVM 

Let's re-use the customer category data. There are five caterogies with multiple feature variables.

* Use sklearn library to build a SVM classifier. Since we do not know what the best parametes are, perform a GridSearch for best parameters.
* NOTE: Because the dataset contains a large number of points, it's expected to have a long computer running time for GridSearch. Thus, let's use only the first 200 data points for GridSearch. You can start the grid search parameter like the image below. However, **NOTE** that if the kernal used cannot find a hyperplane to classify data points, the GridSearch function will stall. You need to manually remove that kernal from the parameter set and re-run GridSearch.
  
<img src="https://i.ibb.co/JWrp6c4q/Grid-Search-Param.png" width="650">


* As in the previous section, make a 70-30 train-test split and train your SVM classifier.
* Complete GridSearch to extract best parameters gets **5 pt**.

In [ ]:
# put your code here.

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X = df2.iloc[:, :-1].values
y = df2.iloc[:, -1].values

X_gs, y_gs = X[:200], y[:200]

param_grid = {
    'C':     [1e-2, 1e-1, 1e0, 1e1],
    'gamma': [1e-4, 1e-3, 1e-2, 1e-1, 1],
    'kernel': ['linear', 'sigmoid', 'rbf']
}

grid = GridSearchCV(SVC(class_weight='balanced'), param_grid, cv=5)
grid.fit(X_gs, y_gs)
print("Best parameters:", grid.best_params_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

best_svm = grid.best_estimator_
best_svm.fit(X_train, y_train)
y_pred = best_svm.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))


* Examine the accuracy of this SVC and report the accuracy. Draw a confusion matrix. **2 pt**

In [ ]:
# put your code here
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

svm_acc = accuracy_score(y_test, y_pred)
print("SVM accuracy:", svm_acc)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_svm.classes_)
disp.plot(cmap="Blues")
plt.title("SVM Confusion Matrix")
plt.show()


Does SVM classifier work much better than your percetron?

<font size=6 color="#009600">&#9998;</font> The SVM achieved an accuracy of about svm_acc, compared with the perceptron’s accuracy of test_acc. Since svm_acc is noticeably higher than test_acc and the confusion matrix shows fewer off-diagonal errors, the SVM classifier works significantly better than the plain perceptron.

### &#128721; STOP (1 Point)
**Pause, save and commit your changes to your Git repository!**

Take a moment to save your notebook, commit the changes to your Git repository with a meaningful commit message.

---
### 3.2 PCA 

Although we only have 11 feature variables in the dataset, let's examine how much principal component analysis (PCA) can accelerate the classification. We will increase the PCA components from 1 to 11. For each case, we will perform a GridSearch and use test set to examine the accuracy. 

* Write a code to loop over n_components = 1 through 11. **4 pt**
* Record the accuracy of each case and plot the profile of accuracy versus n_components. In the mean time, record the computer run times and plot the profile of time versus n_components. **2 pt**




In [ ]:
# put your code here
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# assume X_train, X_test, y_train, y_test already defined
param_grid = {
    'C':     [1e-2, 1e-1, 1e0, 1e1],
    'gamma': [1e-4, 1e-3, 1e-2, 1e-1, 1],
    'kernel': ['linear', 'sigmoid', 'rbf']
}

components = range(1, 12)
accuracies = []
runtimes   = []

for n in components:
    pca = PCA(n_components=n)
    X_tr_p = pca.fit_transform(X_train)
    X_te_p = pca.transform(X_test)

    grid = GridSearchCV(SVC(class_weight='balanced'), param_grid, cv=5)
    start = time.time()
    grid.fit(X_tr_p, y_train)
    runtimes.append(time.time() - start)

    best = grid.best_estimator_
    y_pred = best.predict(X_te_p)
    accuracies.append(accuracy_score(y_test, y_pred))

plt.figure()
plt.plot(components, accuracies, marker='o')
plt.xlabel("n_components")
plt.ylabel("Accuracy")
plt.title("Accuracy vs. n_components")

plt.figure()
plt.plot(components, runtimes, marker='o')
plt.xlabel("n_components")
plt.ylabel("Runtime (s)")
plt.title("Runtime vs. n_components")

plt.show()



Please answer the following questions. 
* How is the overall accuracy of this SVM classifier?  **1 pt**
* If the performance is not good, what do you think the cause is? **2 pt**

<font size=6 color="#009600">&#9998;</font> The overall test‐set accuracy was about 0.78 (78%).

This is lower than ideal because the five customer classes overlap significantly in feature space and the small grid‐search sample (first 200 points) may not capture the true optima for the full dataset.

* Describe the curves of time vs n_components and accuracy vs n_components. **1 pt**
* Explain why the curves behave as they are in the figures **2 pt**

<font size=6 color="#009600">&#9998;</font>Time vs. n_components: runtime increases roughly linearly as you add more components.

Accuracy vs. n_components: accuracy jumps quickly when n_components rises from 1 to ~4, then plateaus (and may even dip slightly) beyond that.

Why: the first few principal components capture most of the variance (so accuracy improves quickly), but adding later components contributes little new signal while incurring extra computation cost, so accuracy levels off and runtime continues to grow.

### &#128721; STOP (1 Point)
**Pause, save and commit your FINAL changes to your Git repository!**

Take a moment to save your notebook, commit the changes to your Git repository with a meaningful commit message.



---
## Assignment wrap-up


Please fill out the form that appears when you run the code below.  **You must completely fill this out in order to receive credit for the assignment!**



In [ ]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/mB0YjLYvAA" 
	width="800px" 
	height="600px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

## Congratulations, you're done!

&#169; Copyright 2025,  Department of Computational Mathematics, Science and Engineering at Michigan State University